# 🛡️ Production-Grade Face Anti-Spoofing Model Training Workflow (CelebA-Spoof)
This notebook handles the entire end-to-end training process for the binary face anti-spoofing classifier on your GPU/ROCm-accelerated training server using the production-grade **CelebA-Spoof** dataset.

### Automated Workflow Steps:
1. **Verify ROCm Hardware**: Checks and initializes PyTorch ROCm/GPU execution.
2. **Dataset Acquisition & Pre-cropped Loading**: Downloads the cropped CelebA-Spoof dataset from Kaggle (`immada/celeba-spoof-cropped`).
3. **Dynamic Path Finding**: Recursively identifies train and test directories automatically.
4. **Data Splitting & Loading**: Structures train/validation datasets and applies robust data augmentations.
5. **Model Compilation**: Imports pre-trained MobileNetV3 and adapts the classifier head to map:
   - Class `0`: `live` (Real)
   - Class `1`: `spoof` (Spoof)
6. **ROCm Training & Validation**: Trains the classifier, logs metric curves, and saves production model weights (`liveness_model.pt`).

In [ ]:
!pip install matplotlib kaggle

In [ ]:
import os
import tarfile
import urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter
import matplotlib.pyplot as plt

print("PyTorch Version:", torch.__version__)
print("ROCm/CUDA GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))

## 1. Automated Production Dataset Acquisition (CelebA-Spoof)
We download the cropped CelebA-Spoof dataset (~1.6GB) from Kaggle using the provided API token. If the download is already complete, we locate the extracted files.

In [ ]:
# Create workspace directories
os.makedirs("liveness_dataset", exist_ok=True)
dataset_dir = "liveness_dataset"
extract_path = os.path.join(dataset_dir, "extracted")

# Pre-configure Kaggle Token environment variables
os.environ["KAGGLE_API_TOKEN"] = "KGAT_ebb30aa62041e8cbdee1192e8901ce62"
try:
    home_kaggle = os.path.expanduser("~/.kaggle")
    os.makedirs(home_kaggle, exist_ok=True)
    with open(os.path.join(home_kaggle, "access_token"), "w") as f:
        f.write("KGAT_ebb30aa62041e8cbdee1192e8901ce62\n")
    os.chmod(os.path.join(home_kaggle, "access_token"), 0o600)
except Exception as e:
    print(f"Kaggle credentials configuration warning: {e}")

# Download CelebA-Spoof cropped dataset from Kaggle
if not os.path.exists(extract_path) or len(os.listdir(extract_path)) == 0:
    print("Downloading production-grade CelebA-Spoof dataset (~1.6GB)... This may take a few minutes...")
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        api.dataset_download_files('immada/celeba-spoof-cropped', path=extract_path, unzip=True)
        print("Download and extraction completed via Kaggle!")
    except Exception as e:
        print(f"Kaggle API download failed: {e}. Check token validity or network connectivity.")
else:
    print("CelebA-Spoof dataset is already downloaded and extracted.")

## 2. Dynamic Path Resolution
We dynamically locate the train and test directories for the loaded CelebA-Spoof dataset.

In [ ]:
train_dir = None
val_dir = None

# Search recursively for the CelebA_Spoof dataset directories
for root, dirs, files in os.walk(extract_path):
    if "train" in dirs and "test" in dirs:
        train_path = os.path.join(root, "train")
        val_path = os.path.join(root, "test")
        
        # Validate sub-directories to ensure they contain class folders like 'live' or 'spoof'
        train_subs = os.listdir(train_path) if os.path.exists(train_path) else []
        if "live" in train_subs or "spoof" in train_subs:
            train_dir = train_path
            val_dir = val_path
            break

if train_dir and val_dir:
    print(f"Successfully located CelebA-Spoof directories:")
    print(f"  - Train Directory: {train_dir}")
    print(f"  - Validation Directory: {val_dir}")
else:
    raise FileNotFoundError("Could not locate standard CelebA-Spoof train/test splits under extraction path.")

## 3. Training Pipeline & Model Definition
We define our data transformations, build PyTorch data loaders, compile our adapted MobileNetV3 classifier head, and execute GPU training.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)

print("Class-to-index mapping:", train_dataset.class_to_idx)
# Ensure 'live' (index 0) and 'spoof' (index 1) match target mapping assumptions
assert train_dataset.class_to_idx['live'] == 0, "Expected class 'live' to map to index 0."
assert train_dataset.class_to_idx['spoof'] == 1, "Expected class 'spoof' to map to index 1."

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

class LivenessModel(nn.Module):
    def __init__(self):
        super(LivenessModel, self).__init__()
        self.backbone = models.mobilenet_v3_small(pretrained=True)
        in_features = self.backbone.classifier[3].in_features
        self.backbone.classifier[3] = nn.Linear(in_features, 2)

    def forward(self, x):
        return self.backbone(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Target training device:", device)

model = LivenessModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

### Run Training Loop (GPU/ROCm Accelerated)
Let's run 5 epochs of training to reach production-grade performance on the CelebA-Spoof dataset.

In [ ]:
epochs = 5
print("Starting production-grade Face Liveness model training...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = (correct / total) * 100
    
    # Validation Loop
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for idx, (inputs, labels) in enumerate(val_loader):
            # Limit validation evaluation during training steps to speed up logging
            if idx >= 100:
                break
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
    val_epoch_loss = val_loss / val_total
    val_acc = (val_correct / val_total) * 100
    
    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.2f}% | "
          f"Val Loss: {val_epoch_loss:.4f} Acc: {val_acc:.2f}%")

# Save model weights to the target locations
save_paths = ["liveness_model.pt"]
try:
    if os.path.basename(os.getcwd()) == "notebooks":
        save_paths.append("../notebooks/liveness_model.pt")
    else:
        save_paths.append("notebooks/liveness_model.pt")
except Exception:
    pass

for p in save_paths:
    try:
        torch.save(model.state_dict(), p)
        print(f"Model saved successfully to: {p}")
    except Exception as e:
        print(f"Could not save to {p}: {e}")

## 5. Visualizing Classifier Predictions
We run inference on validation images to visualize liveness detection classification output.

In [ ]:
# Load one batch from val loader
inputs, labels = next(iter(val_loader))
model.eval()
with torch.no_grad():
    outputs = model(inputs.to(device))
    _, preds = outputs.max(1)

# Helper to un-normalize and show image
def show_img(inp, title=None, color="black"):
    inp = inp.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    plt.imshow(inp)
    if title:
        plt.title(title, color=color, fontsize=10)
    plt.axis("off")

# Plot predictions
fig = plt.figure(figsize=(12, 8))
classes = ["Real", "Spoof"]
num_to_plot = min(6, len(inputs))

for i in range(num_to_plot):
    ax = fig.add_subplot(2, 3, i + 1)
    true_label = classes[labels[i].item()]
    pred_label = classes[preds[i].item()]
    is_correct = (labels[i].item() == preds[i].item())
    
    title = f"True: {true_label}\nPred: {pred_label}"
    color = "green" if is_correct else "red"
    show_img(inputs[i], title=title, color=color)

plt.tight_layout()
plt.show()